# E4b · Inyección-recuperación por exposición

**Spec:** [`docs/spec_E4b_codex_perexp_injection.md`](../docs/spec_E4b_codex_perexp_injection.md)  |  **Bloque:** E · Detección y límites  |  **Run de este set:** `ROXs42Bb_realigned`

Inyecta la fuente sintética en CADA exposición con su propia PSF y combina las medidas, en vez de inyectar una vez en el cubo combinado.

| | |
|---|---|
| **Entrada** | `psf_model_mixture.json` de forma `mixture` (C1 con `psf_scope=per_observation`), `observation_plan.json`, `stage01c_qc.json` y los cubos por exposición |
| **Salida (QC/productos)** | `tables/perexp_injection_by_exposure.csv`, `tables/perexp_injection_combined.csv` y `stages/stage_h04b_qc.json` |
| **Consume aguas abajo** | Ninguna etapa: E3 sigue leyendo a E4. Es producto de referencia y validación, por el mismo motivo que C7. |


## Cómo ejecutar de forma independiente

```bash
conda activate MUSE               # kernel/env con astropy + musepipe
export RUN=ROXs42Bb_realigned   # el run de este objeto
cd MUSE-accretion-pipeline                    # raíz del repo
python -m musepipe.stages.stage_h04b_perexp_injection --run-id $RUN
```

Pesado: la rejilla de E4 sobre cada una de las 29–30 exposiciones.

La celda de abajo hace lo mismo desde el notebook (guardada por `RUN`).


In [ ]:
import os, sys
# Localiza la raíz del repo ascendiendo hasta encontrar `musepipe/` (robusto a
# la profundidad: funciona con el cwd en notebooks/<obj>/, en notebooks/ o en la
# raíz). Añade la raíz (para `import musepipe`) y notebooks/ (para `_nbcommon`).
_d = os.getcwd()
while _d != os.path.dirname(_d):
    if os.path.isdir(os.path.join(_d, 'musepipe')) and os.path.isdir(os.path.join(_d, 'notebooks')):
        break
    _d = os.path.dirname(_d)
_root = _d
for _p in (_root, os.path.join(_root, 'notebooks')):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import _nbcommon as nb
try:
    import matplotlib as mpl
    mpl.rcParams['figure.dpi'] = 120     # retina dobla esto sin agrandar
    mpl.rcParams['savefig.dpi'] = 200
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats('retina')
except Exception:
    pass
RUN_ID = nb.resolve_run_id('ROXs42Bb_realigned')
print('run  =', RUN_ID)
print('root =', _root)
print('dir  =', nb.run_dir(RUN_ID))
print('QC   =', nb.provenance_line('stages/stage_h04b_qc.json', RUN_ID))


## Ejecutar o auditar


In [ ]:
RUN = False   # -> True para RE-EJECUTAR esta etapa (regenera su QC)

if RUN:
    cmd = 'python -m musepipe.stages.stage_h04b_perexp_injection --run-id $RUN'.replace('$RUN', RUN_ID)
    print('ejecutando:', cmd)
    import subprocess
    subprocess.run(cmd, shell=True, cwd=str(nb.project_root()), check=True)
else:
    print('Modo auditoría (RUN=False): se carga el QC existente abajo.')


## QC / resultados


In [ ]:
qc = nb.load_qc_optional('stages/stage_h04b_qc.json', RUN_ID)
nb.show(qc, keys=['convention.sigma', 'convention.methods', 'convention.combine', 'input.n_exposures', 'positions_per_point', 'rows_per_point'], title='E4b')


## Los términos de este QC, en físico

| Término | Qué es | Por qué importa |
|---|---|---|
| `convention.sigma` | De dónde sale la σ que convierte S/N inyectada en flujo. | Tiene que ser la de los controles **de cada exposición**: inyectar «S/N=1» con la σ del combinado en una exposición de 300 s no es S/N=1. |
| `positions_per_point` vs `rows_per_point` | Posiciones de cielo distintas contra filas de tabla. | La unidad independiente es la **posición**; confundirlas infla la significancia (medido el 2026-08-27: Fisher daba p=0.0010 donde solo había 4 posiciones). |
| `convention.methods` | Los extractores medidos por exposición. | `aperture` es el control limpio y `psffit` el que tiene el pedestal medido en el combinado (+659 con señal nula): son los dos que hacen falta para saber si el pedestal es del sustrato o del estimador. |


## Decisiones y notas
- **Etapa aparte, no un knob de E4.** E3 consume la tabla de throughput de E4, así que un interruptor de sustrato dentro de E4 cambiaría en silencio el límite de Ṁ publicado. Es el mismo motivo por el que C7 no entró en `METHOD_ORDER`. · [`docs/spec_E4b_codex_perexp_injection.md`](../docs/spec_E4b_codex_perexp_injection.md)
- **La σ que convierte S/N en flujo es la de cada exposición**, medida en sus propios controles. Usar la del combinado haría que «S/N=1» no fuera S/N=1 en una exposición de 300 s. · [`docs/spec_E4b_codex_perexp_injection.md`](../docs/spec_E4b_codex_perexp_injection.md)
- **Existe porque el sustrato cambia el error del modelo por un factor de 3 a 6**: medido el 2026-08-27, las exposiciones de la noche buena de ROXs 12 b subestiman el halo a la separación del compañero en +55 a +107 %, contra el +17 % del combinado. · [`docs/2026-08-27_modelo_psf_por_exposicion.md`](../docs/2026-08-27_modelo_psf_por_exposicion.md)
